# Глубинное обучение, магистратура НИУ ВШЭ

## Домашнее задание 2. Детекция объектов, YOLO

### Общая информация

Мягкий дедлайн: 30.11.2026 00:00

Жесткий дедлайн: 07.12.2026 00:00

### О задании

Сыграем в квиддич? Или лучше в карты?

В этом задании вам предстоит почти с нуля написать детектор в духе YOLO, а затем обучить готовую модель YOLO из библиотеки `ultralytics` и сравнить результаты. На выбор даются два датасета, отличаются они только картинками. Форматы и баллы одинаковые.

Первый вариант это кадры игры в квиддич из Гарри Поттера. Если вы забыли правила, нажмите [сюда](https://harrypotter.fandom.com/ru/wiki/%D0%9A%D0%B2%D0%B8%D0%B4%D0%B4%D0%B8%D1%87). Вы научитесь находить на кадрах бладжеры, квоффл и снитч.

Второй вариант это датасет с игральными картами. Вы научитесь находить на фотографиях несколько типов карт.

Оба датасета содержат около 300 картинок, разметка хранится в xml в формате Pascal VOC.

Задание состоит из трех частей:
1. Загрузка и визуализация данных.
2. Самописный YOLO-подобный детектор: кодирование разметки, модель, функция потерь, обучение и NMS.
3. Обучение YOLO из `ultralytics` на тех же данных.

Если с самописным детектором совсем не получается, после первой части можно перейти к третьей, с готовой моделью будет проще.

Задание проверялось в Google Colab. Работоспособность в других окружениях не гарантируется.

Консультация 2023 года: https://www.youtube.com/watch?v=oCwEmKW3bYg

### Оценивание и штрафы

Каждая из задач имеет определенную «стоимость» (указана в скобках около задачи). Максимально допустимая оценка за работу без бонуса 10 баллов, за бонус можно получить еще 1 балл.

Сдавать задание после жесткого дедлайна нельзя. Сдача после мягкого дедлайна штрафуется ступенчато: -1 балл за каждые сутки опоздания. Дважды за курс можно сдать работу в жесткий дедлайн без штрафа, заранее предупредив ассистента.

Задание выполняется самостоятельно. «Похожие» решения считаются плагиатом, и все задействованные студенты (в том числе те, у кого списали) не могут получить за него больше 0 баллов. Если вы нашли решение какого-то из заданий (или его часть) в открытом источнике, укажите ссылку на этот источник в отдельном блоке в конце работы (скорее всего, вы будете не единственным, кто это нашел, и ссылка снимет подозрение в плагиате).

Неэффективная реализация кода может негативно отразиться на оценке. Оценка также может быть снижена за плохо читаемый код и плохо оформленные графики. Все ответы должны сопровождаться кодом или комментариями о том, как они были получены.

Использование генеративных моделей допустимо на следующих условиях:
- код, написанный генеративными моделями, составляет не более 30% решения;
- указаны использованная модель и промпты;
- в конце работы описан ваш опыт: как часто приходилось исправлять код руками или просить модель что-то исправить, было ли это быстрее, чем написать код самим.

Если эти условия не выполнены, работа оценивается в 0 баллов.

### Формат сдачи

Сдайте выполненный ноутбук с сохраненными выводами ячеек. Назовите файл `HW2-Username.ipynb`, где Username ваши фамилия и имя на латинице именно в таком порядке (например, `HW2-IvanovIvan.ipynb`).

## Часть 1. Данные (1.5 балла)

Скачайте один из датасетов на свой вкус. Разметка лежит в папке `xmls`, картинки в `images`.

In [ ]:
# Harry Potter
# !wget https://github.com/markblumenau/hw3_iad_dl/raw/main/harry/data.zip

# Cards
# !wget https://github.com/markblumenau/hw3_iad_dl/raw/main/cards/data.zip

!unzip -q data.zip

In [ ]:
import ast
import shutil
from pathlib import Path
from xml.etree import ElementTree as ET

import albumentations as A
import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision
from albumentations.pytorch import ToTensorV2
from matplotlib.patches import Rectangle
from PIL import Image
from torch import nn
from torchvision.models import ResNet50_Weights
from tqdm.notebook import tqdm

**Задание 1.1 (0.5 балла).** Ниже написан `Dataset` из PyTorch. Он требует реализации методов `__getitem__` и `__len__`, которые затем используются для формирования батчей. Разметка хранится в xml, поэтому сначала допишите функцию `get_xml_data`, которая по имени картинки возвращает ее боксы.

В функцию передается `class_dict`, с его помощью название класса превращается в int. Дальше везде подразумевается, что класс записан как int.

Боксы нужны в формате `pascal_voc`, то есть `[xmin, ymin, xmax, ymax, class]`, подробнее в [документации albumentations](https://albumentations.ai/docs/getting_started/bounding_boxes_augmentation/).

In [ ]:
def get_xml_data(image_name, root, class_dict, xml_dir="xmls"):
    xml_path = Path(root) / xml_dir / (Path(image_name).stem + ".xml")
    treeroot = ET.parse(xml_path).getroot()

    bboxes = []
    for member in treeroot.findall("object"):
        # res = [xmin, ymin, xmax, ymax, class], все значения int
        res = ...  # YOUR CODE HERE (－.－)...zzzZZZzzzZZZ
        bboxes.append(res)

    return bboxes

In [ ]:
class PascalDataset(torch.utils.data.Dataset):
    def __init__(self, *, transform, root="data", train=True, seed=42):
        self.root = Path(root)
        self.transform = transform

        assert self.root.is_dir(), f"No data at `{root}`"

        filenames = np.array(sorted(str(p) for p in (self.root / "images").glob("*")))
        with open(self.root / "class_dict") as f:
            self.class_dict = ast.literal_eval(f.readline())
        self.class_dict_inv = {v: k for k, v in self.class_dict.items()}

        permutation = np.random.default_rng(seed).permutation(len(filenames))
        n_train = int(len(filenames) * 0.9)
        idx = permutation[:n_train] if train else permutation[n_train:]
        self.filenames = filenames[idx].tolist()

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        image = ...  # YOUR CODE HERE: RGB np.ndarray (H, W, 3)
        bboxes = ...  # YOUR CODE HERE
        return self.transform(image=image, bboxes=bboxes)

    def get_raw_item(self, idx):
        fname = self.filenames[idx]
        return fname, get_xml_data(fname, self.root, self.class_dict)

    def __len__(self):
        return ...  # YOUR CODE HERE

Ниже определены стандартная нормализация и приведение размера к 512x512.

In [ ]:
mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)

train_transform = A.Compose(
    [
        A.Resize(512, 512),
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ],
    bbox_params=A.BboxParams(format="pascal_voc", min_visibility=0.3),
)

test_transform = A.Compose(
    [
        A.Resize(512, 512),
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ],
    bbox_params=A.BboxParams(format="pascal_voc", min_visibility=0.5),
)

In [ ]:
train_ds = PascalDataset(root="data", transform=train_transform, train=True)
test_ds = PascalDataset(root="data", transform=test_transform, train=False)
len(train_ds), len(test_ds)

**Задание 1.2 (1 балл).** Прежде чем обучать модели, посмотрим на данные. Напишите функцию `visualize`, которая принимает списки изображений и боксов и рисует боксы на изображениях с подписью класса.

В датасете есть `class_dict_inv`, который превращает int с классом обратно в строку с названием.

Полезные функции:
* [plt.subplots](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplots.html) для нескольких изображений в одной фигуре;
* [ax.imshow](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.imshow.html) для отображения картинки (не забудьте откатить нормализацию);
* [ax.text](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.text.html) и [patches.Rectangle](https://matplotlib.org/stable/api/_as_gen/matplotlib.patches.Rectangle.html) для боксов и подписей.

In [ ]:
def visualize(images, bboxes, class_dict_inv=train_ds.class_dict_inv):
    fig, axes = plt.subplots(2, len(images) // 2 + len(images) % 2, figsize=(10, 8), dpi=100)

    for i, ax in enumerate(axes.reshape(-1)):
        ax.axis(False)
        if i >= len(images):
            break

        # Переставьте оси из (C, H, W) в (H, W, C) и откатите нормализацию
        # YOUR CODE HERE (－.－)...zzzZZZzzzZZZ

        for bbox in bboxes[i]:
            # Нарисуйте бокс и подпишите класс
            # YOUR CODE HERE (－.－)...zzzZZZzzzZZZ
            pass

    fig.tight_layout()
    plt.show()

У вас должно получиться что-то похожее на изображения для датасета с масками:

![image](https://i.imgur.com/V5TUT26.png)

In [ ]:
out = [train_ds[i] for i in range(6)]
visualize([o["image"] for o in out], [o["bboxes"] for o in out])

## Часть 2. YOLO-подобный детектор (5 баллов)

Реализуем детектор, похожий на YOLO, один из самых простых детекторов с точки зрения реализации. Он описан в статье [You Only Look Once: Unified, Real-Time Object Detection](https://arxiv.org/abs/1506.02640). Мы его немного упростим: признаки будем извлекать с помощью ResNet50, а на выходе получать карту размера 16x16.

**Задание 2.1 (1 балл).** Реализуйте collate function. Она определяет, как именно батч собирается из отдельных примеров (подробнее в [документации PyTorch](https://pytorch.org/docs/stable/data.html#dataloader-collate-fn)).

Функция принимает список примеров и возвращает картинки и тензор целей размера `Bx6x16x16`. Первая размерность это число примеров в батче, вторая это «каналы», описанные ниже, дальше идет сетка 16 на 16. Порядок координат в сетке зафиксируем как `(y, x)`, то есть в `target` записываем по индексам `cy_idx, cx_idx`.

В шести каналах записаны:
* сдвиги центра бокса относительно левого верхнего угла клетки, деленные на размер клетки («клетка» это гиперпиксель карты 16 на 16), 2 канала (X, Y);
* ширина и высота бокса, деленные на ширину и высоту изображения, 2 канала (W, H);
* confidence: 1 там, где центр бокса попал в клетку, и 0 иначе, 1 канал;
* класс объекта (тот самый int), 1 канал.

Если в одну клетку попали центры двух боксов, оставьте любой из них. Не забудьте про картинки без объектов.

Поясняющая картинка (она схематична: черных клеток должно быть 512, а зеленая должна покрывать 32 черных):

![image](https://i.imgur.com/13YVxAd.jpeg)

In [ ]:
def collate_fn(batch, downsample=32):
    imgs = torch.stack([b["image"] for b in batch])
    batch_boxes = [b["bboxes"] for b in batch]
    b, _, h, w = imgs.shape

    target = imgs.new_zeros(b, 6, h // downsample, w // downsample)

    for i, boxes in enumerate(batch_boxes):
        if len(boxes) == 0:
            continue
        xmin, ymin, xmax, ymax, classes = imgs.new_tensor(boxes).T

        x_cell = ...  # размер клетки по X в пикселях
        y_cell = ...  # размер клетки по Y в пикселях

        w_box = ...  # ширина бокса, деленная на ширину изображения
        h_box = ...  # высота бокса, деленная на высоту изображения

        cx = ...  # центр бокса в пикселях
        cy = ...
        cx_idx = ...  # индекс клетки на карте 16x16, куда попал центр (long)
        cy_idx = ...

        cx_box = ...  # сдвиг центра внутри клетки в долях клетки
        cy_box = ...

        target[i, :, cy_idx, cx_idx] = torch.stack(
            [cx_box, cy_box, w_box, h_box, torch.ones_like(cx_box), classes]
        )

    return {"image": imgs, "target": target}

Проверить реализацию можно функцией ниже. Проверка не блокирующая: в теории корректное решение может ее не пройти.

In [ ]:
def test_collate_fn():
    target1 = [100, 200, 200, 300, 2]  # xmin, ymin, xmax, ymax, class
    target2_1 = [0, 250, 200, 300, 0]
    target2_2 = [0, 100, 100, 300, 1]

    result = collate_fn(
        [
            {"image": torch.rand((3, 512, 512)), "bboxes": [target1]},
            {"image": torch.rand((3, 512, 512)), "bboxes": [target2_1, target2_2]},
            {"image": torch.rand((3, 512, 512)), "bboxes": []},
        ]
    )

    assert result["image"].shape == (3, 3, 512, 512)
    assert result["target"].shape == (3, 6, 16, 16)

    assert np.allclose(
        result["target"][0, :, 7, 4],
        torch.tensor([22 / 32, 26 / 32, 100 / 512, 100 / 512, 1, 2]),
    )
    assert np.allclose(
        result["target"][1, :, 8, 3],
        torch.tensor([4 / 32, 19 / 32, 200 / 512, 50 / 512, 1, 0]),
    )
    assert np.allclose(
        result["target"][1, :, 6, 1],
        torch.tensor([18 / 32, 8 / 32, 100 / 512, 200 / 512, 1, 1]),
    )

    result["target"][0, :, 7, 4] = 0
    result["target"][1, :, 8, 3] = 0
    result["target"][1, :, 6, 1] = 0
    assert np.allclose(result["target"], 0)


test_collate_fn()

Так выглядит сетка 16 на 16 на исходном изображении:

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
i = 20

img = train_ds[i]["image"].permute(1, 2, 0) * torch.tensor(std) + torch.tensor(mean)
bboxes = torch.tensor(train_ds[i]["bboxes"])

ax.imshow(img.clip(0, 1))
loc = plt.matplotlib.ticker.MultipleLocator(base=32)
ax.xaxis.set_major_locator(loc)
ax.yaxis.set_major_locator(loc)
ax.grid(which="major", axis="both", linestyle="-", linewidth=3)

for bbox in bboxes:
    xmin, ymin, xmax, ymax = bbox[:4]
    ax.add_patch(Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, fill=False, color="red"))

cx = (bboxes[:, 0] + bboxes[:, 2]) / 2
cy = (bboxes[:, 1] + bboxes[:, 3]) / 2
ax.scatter(cx, cy, color="green", marker="o")
plt.show()

**Задание 2.2 (0.5 балла).** Выход нашей сети будет больше, чем `Bx6x16x16`. Почему? Классов у нас несколько, и, как в обычной классификации, для каждой клетки сеть будет выдавать вектор длины $C$ из логитов классов. Итого выход имеет размер `Bx(5+C)x16x16`.

Реализуйте преобразование, обратное `collate_fn`, чтобы декодировать выход сети в боксы. Применив `decode_prediction` к выходу `collate_fn`, вы должны получить исходный набор боксов с корректными координатами и классами. Применив ее к выходу сети, тоже должны получить набор боксов с классами.

Если каналов 6, перед нами target, и класс нужно брать из последнего канала. Если каналов больше 6, перед нами выход сети, и класс это argmax по последним $C$ каналам. Возвращайте только боксы, у которых confidence больше `threshold`. Координаты округляйте до int.

In [ ]:
def decode_prediction(pred, upsample=32, threshold=0.7):
    """
    Returns list of length B, result[i] is a list of boxes on image i,
    each box is [xmin, ymin, xmax, ymax, class].
    """
    b, c, h, w = pred.shape
    img_w, img_h = w * upsample, h * upsample
    result = []
    # YOUR CODE HERE (－.－)...zzzZZZzzzZZZ
    return result

Проверка (тоже не блокирующая):

In [ ]:
def test_decode_prediction():
    target1 = [[100, 200, 200, 300, 2]]
    target2 = [[0, 250, 200, 300, 0], [0, 100, 100, 300, 1]]
    result = collate_fn(
        [
            {"image": torch.rand((3, 512, 512)), "bboxes": target1},
            {"image": torch.rand((3, 512, 512)), "bboxes": target2},
        ]
    )

    target = result["target"]
    expected = [target1, [target2[1], target2[0]]]
    expected_alt = [target1, [target2[0], target2[1]]]
    assert decode_prediction(target) in (expected, expected_alt)

    # Имитируем выход сети с тремя классами
    target = torch.cat([target, torch.zeros(2, 2, 16, 16)], dim=1)
    target[0, 5:, 7, 4] = torch.tensor([0.5, 0.6, 0.9])  # класс 2
    target[1, 5:, 8, 3] = torch.tensor([0.9, 0.6, 0.5])  # класс 0
    target[1, 5:, 6, 1] = torch.tensor([0.6, 0.9, 0.5])  # класс 1

    actual = decode_prediction(target)
    print("Actual:  ", actual)

    # Порядок боксов на второй картинке зависит от реализации, оба варианта верные
    print("Expected:", expected)
    assert actual in (expected, expected_alt)


test_decode_prediction()

**Задание 2.3 (1 балл).** Реализуйте модель. Сначала примените первые 4 блока ResNet50 (до `layer4` включительно, можно через `list(model.children())[:-2]`). Затем добавьте несколько блоков `Conv2d`, `BatchNorm2d`, `ReLU`, постепенно уменьшая число каналов до $5+C$, а размер карты до 16 на 16. Например, каналы 2048, 512, 128, 32, 5+C с ядром 3 и паддингом 1, но вариантов много, попробуйте разные. **Последним слоем обязательно должна быть свертка.**

Первые 5 каналов (сдвиги, размеры и confidence) лежат в отрезке $[0, 1]$, поэтому к ним после финальной свертки применим сигмоиду. Последние $C$ каналов оставьте логитами классов без активации.

In [ ]:
C = ...  # число классов в датасете, можно взять из class_dict


class Detector(nn.Module):
    def __init__(self):
        super().__init__()
        model = torchvision.models.resnet50(weights=ResNet50_Weights.DEFAULT)
        # YOUR CODE HERE (－.－)...zzzZZZzzzZZZ

    def forward(self, img):
        # YOUR CODE HERE (－.－)...zzzZZZzzzZZZ
        return

**Задание 2.4 (0.5 балла).** Реализуйте функцию потерь. Она похожа на оригинальную функцию потерь YOLO v1 и состоит из четырех частей:

- `localization_loss`: MSE по сдвигам центра в клетках, где есть объект;
- `box_loss`: MSE по корням из ширины и высоты в клетках, где есть объект;
- `classification_loss`: кросс-энтропия по логитам классов в клетках, где есть объект;
- `confidence_loss`: бинарная кросс-энтропия confidence для **всех** клеток, отдельно для клеток с объектом (вес 1) и без объекта (вес, например, 0.1: таких клеток гораздо больше, но можно экспериментировать).

Для выбора клеток с объектами сделайте маску по каналу confidence в target (пригодится `masked_select` или индексирование булевой маской). Используйте `reduction="sum"` и подумайте, как индексация каналов зависит от $C$.

In [ ]:
def detection_loss(pred, target, C=C):
    # YOUR CODE HERE (－.－)...zzzZZZzzzZZZ
    localization_loss = ...
    box_loss = ...
    classification_loss = ...
    confidence_loss = ...

    return localization_loss + box_loss + classification_loss + confidence_loss

**Задание 2.5 (1 балл).** Обучите модель (напишите цикл обучения) и покажите, что она работает. Скорее всего, объекты найдутся на 1-2 картинках.

In [ ]:
loader = torch.utils.data.DataLoader(train_ds, 10, shuffle=True, collate_fn=collate_fn)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(21)
EPOCHS = ...  # Harry Potter 20, Cards 15
model = Detector().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)

for e in tqdm(range(EPOCHS)):
    epoch_losses = []
    for batch in tqdm(loader, leave=False):
        # YOUR CODE HERE (－.－)...zzzZZZzzzZZZ
        pass

    print(f"Epoch {e} done; Train loss {np.mean(epoch_losses):.3f};")

Запустим обученный детектор на тестовых изображениях:

In [ ]:
test_loader = torch.utils.data.DataLoader(test_ds, 6, collate_fn=collate_fn)
batch = next(iter(test_loader))

In [ ]:
# Сделайте предсказание и переложите результат на cpu
# YOUR CODE HERE (－.－)...zzzZZZzzzZZZ

In [ ]:
# Визуализируйте предсказания. Поиграйтесь с threshold, скорее всего его нужно понизить до ~0.1
# YOUR CODE HERE (－.－)...zzzZZZzzzZZZ

Результат так себе, да? Вариантов улучшения много, самый простой из них это добавить на выход Non-Maximum Suppression.

**Задание 2.6 (1 балл).** Реализуйте NMS сами, без `torchvision.ops.nms`. Напомним принцип работы алгоритма (для каждого класса отдельно):
- сортируем боксы по убыванию уверенности;
- берем самый уверенный бокс и считаем его Intersection over Union (IoU) с остальными;
- выкидываем все боксы, у которых IoU с ним больше порога;
- переходим к следующему оставшемуся боксу, пока список не закончится.

Для этого `decode_prediction_nms` должна возвращать боксы вместе с confidence: `[xmin, ymin, xmax, ymax, class, score]`. Покажите предсказания на тех же тестовых картинках до и после NMS.

In [ ]:
def nms(bboxes, iou_threshold):
    # YOUR CODE HERE (－.－)...zzzZZZzzzZZZ
    return

In [ ]:
def decode_prediction_nms(pred, upsample=32, threshold=0.7, iou_threshold=0.5):
    b, c, h, w = pred.shape
    img_w, img_h = w * upsample, h * upsample
    # YOUR CODE HERE (－.－)...zzzZZZzzzZZZ
    return

In [ ]:
# YOUR CODE HERE (－.－)...zzzZZZzzzZZZ

## Часть 3. YOLO из ultralytics (3.5 балла)

Займемся более простыми вещами: возьмем готовую архитектуру, обучим ее на наших данных и посмотрим, что получится. Будем использовать YOLO11 из библиотеки [ultralytics](https://docs.ultralytics.com/).

In [ ]:
!pip install ultralytics

**Задание 3.1 (1.5 балла).** Чтобы модель обучалась одной строкой, данные нужно переложить в правильный формат. Да-да, классика перекладывания JSON. Формат описан в [документации ultralytics](https://docs.ultralytics.com/datasets/detect/).

Если коротко:
* есть `.yaml`, в котором лежат пути к папкам с картинками, число классов и их названия;
* есть папки `train` и `valid` (их соберем за вас), в каждой две подпапки:
    - `images` с картинками;
    - `labels` с txt-файлами, названными как картинки, внутри по строке на бокс: `class x_center y_center width height`, координаты нормированы на размер картинки.

Реализуйте функцию, которая принимает разметку в исходном формате и возвращает список строк в формате YOLO, которые можно сразу записать в файл, соединив через `\n`. Сторонние конвертеры использовать нельзя, переложите руками.

In [ ]:
def annotation2txt(bboxes, w_im, h_im):
    # YOUR CODE HERE (－.－)...zzzZZZzzzZZZ
    return

In [ ]:
yolo_root = Path("yolo_data")
shutil.rmtree(yolo_root, ignore_errors=True)

for split, ds in [("train", train_ds), ("valid", test_ds)]:
    (yolo_root / split / "images").mkdir(parents=True)
    (yolo_root / split / "labels").mkdir(parents=True)
    for i in range(len(ds)):
        fname, bboxes = ds.get_raw_item(i)
        shutil.copyfile(fname, yolo_root / split / "images" / Path(fname).name)
        w_im, h_im = Image.open(fname).size
        with open(yolo_root / split / "labels" / (Path(fname).stem + ".txt"), "w") as f:
            f.write("\n".join(annotation2txt(bboxes, w_im, h_im)))

In [ ]:
nc = ...  # число классов, можно посчитать по class_dict
names = ...  # список названий классов в порядке их id, можно посчитать по class_dict

with open("data.yaml", "w") as f:
    f.write(
        f"""path: {yolo_root.resolve()}
train: train/images
val: valid/images

nc: {nc}
names: {names}
"""
    )

**Задание 3.2 (1.5 балла).** Обучите YOLO11 самого маленького размера (`n`). Библиотека максимально дружелюбная, от вас требуется пара строк. Модель можно взять предобученную.

Подсказка: подумайте, зачем вам `data.yaml`, и чем отличаются `yolo11n.pt` и `yolo11n.yaml`.

Если у вас лосс NaN или Windows, вот несколько полезных ссылок (покрывают не все проблемы, но могут помочь):

https://github.com/ultralytics/ultralytics/issues/1149

https://github.com/ultralytics/yolov5/issues/6907

https://stackoverflow.com/questions/75178762/i-got-nan-for-all-losses-while-training-yolov8-model

In [ ]:
# YOUR CODE HERE (－.－)...zzzZZZzzzZZZ

**Задание 3.3 (0.5 балла).** Отрисуйте предсказания на валидационной выборке (хотя бы на 5-10 картинках).

Можно использовать `save=True` у `predict` и потом прочитать сохраненные картинки, а можно отрисовать боксы из результата `predict` своей функцией `visualize`. Стандартный `show` пытается показывать через `cv2.imshow`, в Colab он работать не будет.

In [ ]:
# YOUR CODE HERE (－.－)...zzzZZZzzzZZZ

## Бонус (1 балл)

Реализуйте подсчет mAP для самописной модели. Если найдете готовый код (например, `MeanAveragePrecision` из `torchmetrics`), можно использовать его со ссылкой на источник.

Посчитайте mAP@0.5 и mAP@0.5:0.95 для своей модели с NMS и для YOLO11 на одной и той же тестовой выборке. Сравните модели. Что получается и почему?

In [ ]:
# YOUR CODE HERE (－.－)...zzzZZZzzzZZZ

**Ответ:** ...

### Источники и генеративные модели

Перечислите источники, которыми пользовались, а также модели и промпты, если использовали генеративный ИИ. Опишите, насколько это помогло.

**Ответ:** ...

Вставьте что угодно, описывающее ваши впечатления от этого задания: